# Lab 4: Autonomous Medical Voice AI Assistant

## Objective
Build an educational Medical Information Voice Assistant using:
- **LangChain & OpenRouter**: Function calling and reasoning.
- **MedlinePlus NIH API**: Live search tool for verified medical topics.
- **Edge-TTS & Whisper**: Speech synthesis and audio transcription.
- **Safety Guardrails**: Strict non-diagnostic, non-prescriptive spoken guidance.


## 1. Load Environment & OpenRouter API Key
Verify that `OPENROUTER_API_KEY` is loaded from the local `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "nvidia/nemotron-3.5-lightning:free")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print(f"OpenRouter API key loaded successfully. Model: {LLM_MODEL}")

## 2. Initialize Chat Model
Initialize `ChatOpenRouter` with automatic fallback to `ChatOpenAI`.

In [ ]:
try:
    from langchain_openrouter import ChatOpenRouter
    model = ChatOpenRouter(
        model=LLM_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0
    )
except ImportError:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(
        model=LLM_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0
    )

print("Model initialized successfully")

## 3. Test Baseline Model Knowledge
Ask a general medical question before attaching tools.

In [ ]:
response = model.invoke(
    "Explain hypertension in simple words."
)

print(response.content)

## 4. Define MedlinePlus Live Medical Search Tool
Connect to the National Library of Medicine (MedlinePlus) API to fetch verified health summaries.

In [ ]:
import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

FALLBACK_TOPICS = {
    "hypertension": "Hypertension, or high blood pressure, occurs when the force of blood against artery walls is consistently too high. Regular exercise, balanced low-salt diet, and routine monitoring are key management steps.",
    "fever": "A fever is a temporary elevation of body temperature above normal, usually as the body fights an infection. Rest and fluids help manage symptoms.",
    "diabetes": "Diabetes is a metabolic condition affecting how the body processes glucose. Nutrition, activity, and medical guidance are essential."
}

@tool
def medical_information(topic: str) -> str:
    """
    Search MedlinePlus for general medical information about a topic.
    Provides educational information only and does not diagnose or prescribe.
    """
    topic_clean = topic.strip().lower()
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {
        "db": "healthTopics",
        "term": topic_clean,
        "retmax": 3,
        "rettype": "brief"
    }

    try:
        import urllib3
        urllib3.disable_warnings()
        response = requests.get(url, params=params, timeout=4, verify=False)
        response.raise_for_status()

        if response.text.strip().startswith("<?xml") or "<nlmSearchResult" in response.text:
            root = ET.fromstring(response.text)
            results = []
            for document in root.findall(".//document"):
                title = ""
                summary = ""
                page_url = document.attrib.get("url", "")
                for content in document.findall("content"):
                    name = content.attrib.get("name")
                    text = "".join(content.itertext()).strip()
                    if name == "title":
                        title = text
                    elif name == "full-summary":
                        summary = text
                if title or summary:
                    results.append({"title": title, "summary": summary, "url": page_url})
            if results:
                output = f"Medical information from MedlinePlus for '{topic}':\n\n"
                for i, result in enumerate(results, 1):
                    output += f"{i}. {result['title']}\n{result['summary']}\nSource: {result['url']}\n\n"
                output += "Important: This information is for educational purposes only. It does not provide a diagnosis or medical prescription."
                return output
    except Exception:
        pass

    for key, text in FALLBACK_TOPICS.items():
        if key in topic_clean or topic_clean in key:
            return f"Medical information for '{topic}':\n\n{text}\n\nImportant: Educational purposes only. No diagnosis or prescription."

    return f"General health overview for '{topic}': Consult trusted sources such as MedlinePlus or a medical professional.\nImportant: Educational purposes only."

## 5. Bind Tools to Model & Test Tool Calling

In [ ]:
model_with_tools = model.bind_tools([medical_information])

response = model_with_tools.invoke(
    "Can you tell me about hypertension?"
)

print("Tool calls detected:", response.tool_calls)

## 6. Speech Recognition Setup (Faster-Whisper)
Load the local Whisper model for transcribing microphone recordings.

In [ ]:
try:
    from faster_whisper import WhisperModel
    whisper_model = WhisperModel("small", device="cpu", compute_type="int8")
    print("Whisper model loaded successfully")
except Exception as e:
    print(f"Whisper model loading skipped or optional: {e}")

## 7. End-to-End Voice Medical Assistant Pipeline
Takes patient input, queries MedlinePlus, enforces strict educational safety guardrails, and speaks the response aloud using Edge Neural TTS.

In [ ]:
import asyncio
import edge_tts
from IPython.display import Audio, display
from langchain_core.messages import HumanMessage, ToolMessage

user_text = input("🧑 Patient: ") or "What causes a fever?"
print(f"\nPatient asked: {user_text}")

system_prompt = """
You are a medical information voice assistant.
Your job is to provide general educational medical information.

Important rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not replace a healthcare professional.
- Use information retrieved from the medical information tool.
- Since your response will be spoken aloud, keep the answer concise.
- Use simple language.
- Do not use Markdown (no asterisks, headings, or bullet points).
- Avoid long lists.
"""

messages = [
    HumanMessage(content=system_prompt),
    HumanMessage(content=user_text)
]

response = model_with_tools.invoke(messages)

if response.tool_calls:
    messages.append(response)
    print("\n🔧 Tool Calling...")
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        print(f"Tool: {tool_name}")
        print(f"Arguments: {tool_args}")

        if tool_name == "medical_information":
            tool_result = medical_information.invoke(tool_args)
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    final_response = model_with_tools.invoke(messages)
    answer = final_response.content
else:
    answer = response.content

print(f"\n🩺 Medical Assistant:\n{answer}")

# Synthesize Speech via Edge-TTS
async def generate_voice(text, output_file="medical_response.mp3", voice="en-IN-NeerjaNeural"):
    communicate = edge_tts.Communicate(text=text, voice=voice)
    await communicate.save(output_file)

await generate_voice(answer)
print("\n🔊 Medical Assistant Voice:")
display(Audio("medical_response.mp3", autoplay=True))
